In [1]:
import eval_beir_helpers
from redis_bm25 import gather_bm25_results
from redis_vector import gather_res_vector
from beir.retrieval.evaluation import EvaluateRetrieval

/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/robert.shelton/.pyenv/versions/3.11.9/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Eval beir datasets against Redis with different search techniques

### Load dataset

In [2]:
from redisvl.utils.vectorize import HFTextVectorizer

# set vectorizer
emb_model = HFTextVectorizer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
# set dataset of interest
dataset = "scidocs"

corpus, queries, qrels = eval_beir_helpers.get_beir_dataset(dataset)
corpus_data = eval_beir_helpers.process_corpus(corpus, emb_model=emb_model)

  0%|          | 0/25657 [00:00<?, ?it/s]

In [4]:
index = eval_beir_helpers.create_redis_index(dataset)

In [5]:
# load if need be
index.load(corpus_data)

['scidocs:01JQ7EXRBWC57DZ39204XZW6MQ',
 'scidocs:01JQ7EXRBWB5AZWXW4JFE9GJN2',
 'scidocs:01JQ7EXRBWB7GA6FTWWM8Q9TBX',
 'scidocs:01JQ7EXRBWTP6MYJ8MSGH32P10',
 'scidocs:01JQ7EXRBWM7HF623Y8MTN05PV',
 'scidocs:01JQ7EXRBWJGZ29JDDAC353707',
 'scidocs:01JQ7EXRBWTEWFA8D8BQDEH1FF',
 'scidocs:01JQ7EXRBW5ATPSATAZ1X2GKQN',
 'scidocs:01JQ7EXRBWSSX16X8W1NE77EB5',
 'scidocs:01JQ7EXRBWEG3RHFEAHVENZ833',
 'scidocs:01JQ7EXRBWY1KXR2YJA9FXAMPJ',
 'scidocs:01JQ7EXRBWZRK729PEV7RT4X8Z',
 'scidocs:01JQ7EXRBWDMRAJPYDNS8QJS9E',
 'scidocs:01JQ7EXRBW48HP0SNWMECC7QDF',
 'scidocs:01JQ7EXRBWGEMF1EFWACJMD6B7',
 'scidocs:01JQ7EXRBW2B3F53S46N3V1G36',
 'scidocs:01JQ7EXRBWPHK6BBXBRPR7XSHA',
 'scidocs:01JQ7EXRBWQYGM5C3NT4D4N98C',
 'scidocs:01JQ7EXRBWM9KGDHVRH9FVBPM7',
 'scidocs:01JQ7EXRBW7V1TPB3VPE6GGB8A',
 'scidocs:01JQ7EXRBWPEG5DBN69500XDM2',
 'scidocs:01JQ7EXRBWV9X5393SNH7BE3HZ',
 'scidocs:01JQ7EXRBWG49RY03529WYHG0X',
 'scidocs:01JQ7EXRBWNRC2RHDPR44WTR5H',
 'scidocs:01JQ7EXRBWXXX8QT5H8MA5S5DM',
 'scidocs:01JQ7EXRBW4H3P5

### Check index info

See if all docs indexed, if there were any indexing errors, and view the total indexing time.

In [6]:
num_docs = index.info()["num_docs"]
total_indexing_time = round(float(index.info()["total_indexing_time"]) / 1000, 3) # indexing time in ms
indexing_failures = index.info()['Index Errors']

print(f"{num_docs=}, {total_indexing_time=} s, {indexing_failures=}")

num_docs=25657, total_indexing_time=29.824 s, indexing_failures=['indexing failures', 0, 'last indexing error', 'N/A', 'last indexing error key', 'N/A']


## Eval for different search methods

In [7]:
# simple bm25

redis_res_bm25 = gather_bm25_results(queries, index)
redis_bm25_ndcg, _map, redis_bm25_recall, redis_bm25_precision = (
    EvaluateRetrieval.evaluate(qrels, redis_res_bm25, [1, 10])
)

print(f"{redis_bm25_ndcg=} \n {redis_bm25_recall=} \n {redis_bm25_precision=}")

failed for 2e08d3ec08069ef41059cc3da6bc4d390eaf6e01, Biases in social comparisons : Optimism or pessimism ? q
failed for 46fe01be6684b2c6d04298f0e40589eff560af2b, Does Mindfulness Meditation Enhance Attention ? A Randomized Controlled Trial
failed for feef5abe52b21ffc198a9f439ec49151c545ef12, Evolvability : What Is It and How Do We Get It ?
failed for b7d0e2a5229d4ec65182c783259a2eeb67a6a873, Who are these people ? ” Evaluating the demographic characteristics and political preferences of MTurk survey respondents
failed for e6be9c497285ece7f32b486c6cc72193b5a1fcb9, Anonymous Post-Quantum Cryptocash ? ( Full Version )
failed for 763490c13a138523b98e7646ef0fa81d6b5e5a22, Normalizing SMS: are Two Metaphors Better than One ?
redis_bm25_ndcg={'NDCG@1': 0.16, 'NDCG@10': 0.13858} 
 redis_bm25_recall={'Recall@1': 0.03245, 'Recall@10': 0.14547} 
 redis_bm25_precision={'P@1': 0.16, 'P@10': 0.0716}


In [8]:
# basic vector

redis_res_vector = gather_res_vector(queries, index, emb_model)
vec_ndcg, _map, vec_recall, vec_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_vector, [1, 10]
)

print(f"{vec_ndcg=} \n {vec_recall=} \n {vec_precision=}")

vec_ndcg={'NDCG@1': 0.231, 'NDCG@10': 0.2105} 
 vec_recall={'Recall@1': 0.04673, 'Recall@10': 0.22517} 
 vec_precision={'P@1': 0.231, 'P@10': 0.1111}


In [9]:
# lin combo
from redis_lin_combo import gather_lin_combo_results

alpha = 0.7
redis_res_lin_combo = gather_lin_combo_results(queries, index, alpha, emb_model)
lin_combo_ndcg, _map, lin_combo_recall, lin_combo_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_lin_combo, [1, 10]
)

print(f"{lin_combo_ndcg=} \n {lin_combo_recall=} \n {lin_combo_precision=}")

failed for 2e08d3ec08069ef41059cc3da6bc4d390eaf6e01, Biases in social comparisons : Optimism or pessimism ? q
failed for 46fe01be6684b2c6d04298f0e40589eff560af2b, Does Mindfulness Meditation Enhance Attention ? A Randomized Controlled Trial
failed for feef5abe52b21ffc198a9f439ec49151c545ef12, Evolvability : What Is It and How Do We Get It ?
failed for b7d0e2a5229d4ec65182c783259a2eeb67a6a873, Who are these people ? ” Evaluating the demographic characteristics and political preferences of MTurk survey respondents
failed for e6be9c497285ece7f32b486c6cc72193b5a1fcb9, Anonymous Post-Quantum Cryptocash ? ( Full Version )
failed for 763490c13a138523b98e7646ef0fa81d6b5e5a22, Normalizing SMS: are Two Metaphors Better than One ?
lin_combo_ndcg={'NDCG@1': 0.207, 'NDCG@10': 0.20333} 
 lin_combo_recall={'Recall@1': 0.04213, 'Recall@10': 0.22967} 
 lin_combo_precision={'P@1': 0.207, 'P@10': 0.1134}


In [10]:
# weighted rrf
from redis_weighted_rrf import gather_weighted_rrf

redis_res_w_rrf = gather_weighted_rrf(queries, index, emb_model)
w_rrf_ndcg, _map, w_rrf_recall, w_rrf_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_w_rrf, [1, 10]
)

print(f"{w_rrf_ndcg=} \n {w_rrf_recall=} \n {w_rrf_precision=}")

failed for 2e08d3ec08069ef41059cc3da6bc4d390eaf6e01, Biases in social comparisons : Optimism or pessimism ? q
failed for 46fe01be6684b2c6d04298f0e40589eff560af2b, Does Mindfulness Meditation Enhance Attention ? A Randomized Controlled Trial
failed for feef5abe52b21ffc198a9f439ec49151c545ef12, Evolvability : What Is It and How Do We Get It ?
failed for b7d0e2a5229d4ec65182c783259a2eeb67a6a873, Who are these people ? ” Evaluating the demographic characteristics and political preferences of MTurk survey respondents
failed for e6be9c497285ece7f32b486c6cc72193b5a1fcb9, Anonymous Post-Quantum Cryptocash ? ( Full Version )
failed for 763490c13a138523b98e7646ef0fa81d6b5e5a22, Normalizing SMS: are Two Metaphors Better than One ?
w_rrf_ndcg={'NDCG@1': 0.2, 'NDCG@10': 0.19013} 
 w_rrf_recall={'Recall@1': 0.04055, 'Recall@10': 0.20797} 
 w_rrf_precision={'P@1': 0.2, 'P@10': 0.1026}


In [11]:
# weighted rrf
from redis_rerank import gather_rerank_results

redis_res_rerank = gather_rerank_results(queries, index, emb_model)
rerank_ndcg, _map, rerank_recall, rerank_precision = EvaluateRetrieval.evaluate(
    qrels, redis_res_rerank, [1, 10]
)

print(f"{rerank_ndcg=} \n {rerank_recall=} \n {rerank_precision=}")

failed for 2e08d3ec08069ef41059cc3da6bc4d390eaf6e01, Biases in social comparisons : Optimism or pessimism ? q
failed for 46fe01be6684b2c6d04298f0e40589eff560af2b, Does Mindfulness Meditation Enhance Attention ? A Randomized Controlled Trial
failed for feef5abe52b21ffc198a9f439ec49151c545ef12, Evolvability : What Is It and How Do We Get It ?
failed for b7d0e2a5229d4ec65182c783259a2eeb67a6a873, Who are these people ? ” Evaluating the demographic characteristics and political preferences of MTurk survey respondents
failed for e6be9c497285ece7f32b486c6cc72193b5a1fcb9, Anonymous Post-Quantum Cryptocash ? ( Full Version )
failed for 763490c13a138523b98e7646ef0fa81d6b5e5a22, Normalizing SMS: are Two Metaphors Better than One ?
rerank_ndcg={'NDCG@1': 0.181, 'NDCG@10': 0.17464} 
 rerank_recall={'Recall@1': 0.03678, 'Recall@10': 0.19917} 
 rerank_precision={'P@1': 0.181, 'P@10': 0.0982}


# Save outputs of run

In [12]:
import pandas as pd

res_data = {
    "Model": ["redis_BM25", "redis_vector", "lin_combo", "weighted_rrf", "rerank"],
    "NDCG@1": [redis_bm25_ndcg['NDCG@1'], vec_ndcg['NDCG@1'], lin_combo_ndcg['NDCG@1'], w_rrf_ndcg['NDCG@1'], rerank_ndcg['NDCG@1']],
    "NDCG@10": [redis_bm25_ndcg['NDCG@10'], vec_ndcg['NDCG@10'], lin_combo_ndcg['NDCG@10'], w_rrf_ndcg['NDCG@10'], rerank_ndcg['NDCG@10']],
    "Recall@1": [redis_bm25_recall['Recall@1'], vec_recall['Recall@1'], lin_combo_recall['Recall@1'], w_rrf_recall['Recall@1'], rerank_recall['Recall@1']],
    "Recall@10": [redis_bm25_recall['Recall@10'], vec_recall['Recall@10'], lin_combo_recall['Recall@10'], w_rrf_recall['Recall@10'], rerank_recall['Recall@10']],
    "Precision@1": [redis_bm25_precision['P@1'], vec_precision['P@1'], lin_combo_precision['P@1'], w_rrf_precision['P@1'], rerank_precision['P@1']],
    "Precision@10": [redis_bm25_precision['P@10'], vec_precision['P@10'], lin_combo_precision['P@10'], w_rrf_precision['P@10'], rerank_precision['P@10']]
}

df = pd.DataFrame(res_data)

In [13]:
df.to_csv(f"{dataset}_results.csv", index=False)

## Cleanup

In [15]:
index.delete()